# Sleep Health & Lifestyle — Exploratory Data Analysis

Comprehensive analysis of sleep patterns, lifestyle factors, and health outcomes across **100,000 individuals**.  
Dataset contains **32 features** spanning demographics, sleep metrics, lifestyle habits, mental health, and cognitive performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/sleep_health_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Dataset Overview

In [ ]:
print(df.info())
print('\n--- Summary Statistics ---')
df.describe().round(2)

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nAvg Sleep Duration: {df["sleep_duration_hrs"].mean():.2f} hrs')
print(f'Avg Sleep Quality: {df["sleep_quality_score"].mean():.2f} / 10')
print(f'Avg Stress Score: {df["stress_score"].mean():.2f} / 10')
print(f'Felt Rested: {df["felt_rested"].mean()*100:.1f}%')
print(f'\nSleep Disorder Risk:')
print(df['sleep_disorder_risk'].value_counts())

## 2. Sleep Duration & Quality Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['sleep_duration_hrs'], bins=40, kde=True, ax=axes[0])
axes[0].axvline(df['sleep_duration_hrs'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {df["sleep_duration_hrs"].mean():.2f}')
axes[0].set_title('Sleep Duration Distribution', fontsize=14, fontweight='bold')
axes[0].legend()

sns.histplot(df['sleep_quality_score'], bins=30, kde=True, ax=axes[1], color='coral')
axes[1].axvline(df['sleep_quality_score'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {df["sleep_quality_score"].mean():.2f}')
axes[1].set_title('Sleep Quality Distribution', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout(); plt.show()

## 3. Sleep Duration vs Quality

In [ ]:
sample = df.sample(5000, random_state=42)
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(sample['sleep_duration_hrs'], sample['sleep_quality_score'],
                c=sample['stress_score'], cmap='RdYlGn_r', alpha=0.5, s=20, edgecolors='w', linewidth=0.2)
plt.colorbar(sc, label='Stress Score')
r = df['sleep_duration_hrs'].corr(df['sleep_quality_score'])
ax.set_title(f'Sleep Duration vs Quality (r = {r:.3f})', fontsize=16, fontweight='bold')
ax.set_xlabel('Sleep Duration (hrs)'); ax.set_ylabel('Sleep Quality Score')
plt.tight_layout(); plt.show()

## 4. Sleep Disorder Risk Analysis

In [ ]:
risk_order = ['Healthy', 'Mild', 'Moderate', 'Severe']
risk_counts = df['sleep_disorder_risk'].value_counts().reindex(risk_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_risk = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[0].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%',
            colors=colors_risk, startangle=140)
axes[0].set_title('Disorder Risk Distribution', fontsize=14, fontweight='bold')

# Risk by age group
df['age_group'] = pd.cut(df['age'], bins=[17,25,35,45,55,70],
                         labels=['18-25','26-35','36-45','46-55','56-69'])
ct = pd.crosstab(df['age_group'], df['sleep_disorder_risk'], normalize='index') * 100
ct[risk_order].plot(kind='bar', stacked=True, ax=axes[1], color=colors_risk)
axes[1].set_title('Disorder Risk by Age Group', fontsize=14, fontweight='bold')
axes[1].set_ylabel('%'); axes[1].legend(title='Risk')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 5. Lifestyle Factors Impact

In [ ]:
# Caffeine
caff_qual = df.groupby('caffeine_mg_before_bed')['sleep_quality_score'].mean().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(caff_qual.index.astype(str), caff_qual.values,
            color=sns.color_palette('YlOrRd', len(caff_qual)))
axes[0].set_title('Sleep Quality by Caffeine Intake', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Caffeine (mg)'); axes[0].set_ylabel('Avg Quality')

# Screen time bins
df['screen_bin'] = pd.cut(df['screen_time_before_bed_mins'],
                          bins=[0,30,60,90,120,180], labels=['0-30','30-60','60-90','90-120','120-180'])
screen_qual = df.groupby('screen_bin', observed=True)['sleep_quality_score'].mean()
axes[1].bar(screen_qual.index.astype(str), screen_qual.values,
            color=sns.color_palette('Blues_r', len(screen_qual)))
axes[1].set_title('Sleep Quality by Screen Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Screen Time (mins)'); axes[1].set_ylabel('Avg Quality')

plt.tight_layout(); plt.show()

## 6. Mental Health & Cognitive Performance

In [ ]:
mh_order = ['Healthy', 'Anxiety', 'Depression', 'Both']
colors_mh = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, col in enumerate(['sleep_duration_hrs', 'sleep_quality_score', 'cognitive_performance_score']):
    mh_data = df.groupby('mental_health_condition')[col].mean().reindex(mh_order)
    axes[i].bar(mh_data.index, mh_data.values, color=colors_mh)
    axes[i].set_title(f'Avg {col.replace("_"," ").title()}', fontsize=12, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
    for j, v in enumerate(mh_data.values):
        axes[i].text(j, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('Mental Health Impact on Sleep & Cognition', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Correlation Heatmap

In [ ]:
num_cols = ['sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage',
            'deep_sleep_percentage', 'sleep_latency_mins', 'wake_episodes_per_night',
            'caffeine_mg_before_bed', 'alcohol_units_before_bed',
            'screen_time_before_bed_mins', 'stress_score', 'work_hours_that_day',
            'heart_rate_resting_bpm', 'bmi', 'steps_that_day',
            'cognitive_performance_score', 'room_temperature_celsius']

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Cognitive Performance vs Sleep Quality

In [ ]:
sample3 = df.sample(5000, random_state=7)
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(sample3['sleep_quality_score'], sample3['cognitive_performance_score'],
                c=sample3['sleep_duration_hrs'], cmap='viridis', alpha=0.5, s=20,
                edgecolors='w', linewidth=0.2)
plt.colorbar(sc, label='Sleep Duration (hrs)')
r_cog = df['sleep_quality_score'].corr(df['cognitive_performance_score'])
z = np.polyfit(df['sleep_quality_score'], df['cognitive_performance_score'], 1)
xs = np.linspace(1, 10, 100)
ax.plot(xs, np.polyval(z, xs), 'r--', lw=2, label=f'Trend (r={r_cog:.3f})')
ax.set_title('Sleep Quality vs Cognitive Performance', fontsize=16, fontweight='bold')
ax.set_xlabel('Sleep Quality Score'); ax.set_ylabel('Cognitive Performance Score')
ax.legend(); plt.tight_layout(); plt.show()

## 9. Key Findings

1. **Average sleep duration is 6.42 hours** — below the recommended 7-9 hours
2. **Only 39% of people felt rested** after sleep
3. **54.2% have Healthy sleep disorder risk**, while 4.1% face Severe risk
4. **Strong correlation between sleep quality and cognitive performance** — better sleep = better brain function
5. **Higher caffeine intake before bed reduces sleep quality** — 400mg group scores lowest
6. **Mental health conditions (especially 'Both') significantly reduce** sleep duration, quality, and cognitive scores
7. **Higher stress correlates with lower sleep quality**
8. **Screen time before bed negatively impacts sleep quality** — 120-180 min group scores worst
9. **Exercise days show slightly better sleep metrics** than non-exercise days
10. **Morning chronotypes** tend to report slightly better sleep outcomes